In [1]:
!pip install pandas numpy pyarrow duckdb

In [2]:
import pandas as pd
import numpy as np

from datetime import datetime,timedelta

np.random.seed(42)

n = 500000
user_id = np.arange(1, n + 1)
cities = ["Berlin", "Tokyo", "New York", "London", 
          "Paris", "Toronto", "Sydney", 
          "San Francisco", "Singapore", "Dubai"]

city = np.random.choice(cities, n)
score = np.round(np.random.uniform(0, 100, n), 2)
active = np.random.choice([True, False], n)

age = np.random.randint(18, 81, n)
sessions = np.random.randint(0, 501, n)
revenue = np.round(np.random.uniform(0, 1000, n), 2)

today = datetime.today()
start_date = today - timedelta(3*360)
signup_date = pd.to_datetime(
    np.random.randint(
        int(start_date.timestamp()),
        int(today.timestamp()),
        n
    ),
    unit="s"
)

df = pd.DataFrame({
    "user_id": user_id,
    "city": city,
    "score": score,
    "active": active,
    "signup_date": signup_date,
    "age": age,
    "sessions": sessions,
    "revenue": revenue
})

In [3]:
print(df.head())
print(df.describe())
print(df.info())

   user_id           city  score  active         signup_date  age  sessions  \
0        1         Sydney  43.69   False 2025-07-06 10:42:07   50       402   
1        2         London  97.40    True 2024-10-13 12:21:40   63       199   
2        3  San Francisco  66.15   False 2023-10-10 09:10:13   54       442   
3        4          Paris  21.99   False 2024-04-24 17:03:26   38       187   
4        5         Sydney  91.72    True 2025-04-03 15:05:44   71       478   

   revenue  
0   497.10  
1   630.01  
2   426.60  
3   693.52  
4   425.30  
             user_id          score          signup_date            age  \
count  500000.000000  500000.000000               500000  500000.000000   
mean   250000.500000      50.063230  2024-09-11 21:28:43      48.997940   
min         1.000000       0.000000  2023-03-21 09:27:32      18.000000   
25%    125000.750000      25.090000  2023-12-17 03:17:46      33.000000   
50%    250000.500000      50.090000  2024-09-11 16:34:59      49.000000 

In [4]:
df.to_parquet("data.parquet", index=False)


In [5]:
import pyarrow as pa
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile("data.parquet")
print("Number of row groups:", parquet_file.metadata.num_row_groups)
print("Number of rows:", parquet_file.metadata.num_rows)
print("Number of columns:", parquet_file.metadata.num_columns)
print("Schema:", parquet_file.schema)

Number of row groups: 1
Number of rows: 500000
Number of columns: 8
Schema: <pyarrow._parquet.ParquetSchema object at 0x137761040>
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional double field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=milliseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 age;
  optional int64 field_id=-1 sessions;
  optional double field_id=-1 revenue;
}



In [6]:
row_group = parquet_file.metadata.row_group(0)
for i in range(row_group.num_columns):
    col = row_group.column(i)
    print(f"Column: {col.path_in_schema:10s} | "
          f"Type: {col.physical_type:8s} | "
          f"Compressed size: {col.total_compressed_size:>8d} bytes")

    statistics = col.statistics
    if statistics is not None:
        print("  Has statistics:", True)
        print("  Min:", statistics.min)
        print("  Max:", statistics.max)
        print("  Null count:", statistics.null_count)
    else:
        print("  Has statistics:", False)

Column: user_id    | Type: INT64    | Compressed size:  2273849 bytes
  Has statistics: True
  Min: 1
  Max: 500000
  Null count: 0
Column: city       | Type: BYTE_ARRAY | Compressed size:   252624 bytes
  Has statistics: True
  Min: Berlin
  Max: Toronto
  Null count: 0
Column: score      | Type: DOUBLE   | Compressed size:   925057 bytes
  Has statistics: True
  Min: -0.0
  Max: 100.0
  Null count: 0
Column: active     | Type: BOOLEAN  | Compressed size:    63800 bytes
  Has statistics: True
  Min: False
  Max: True
  Null count: 0
Column: signup_date | Type: INT64    | Compressed size:  3737692 bytes
  Has statistics: True
  Min: 2023-03-21 09:27:32
  Max: 2026-03-05 09:25:10
  Null count: 0
Column: age        | Type: INT64    | Compressed size:   378359 bytes
  Has statistics: True
  Min: 18
  Max: 80
  Null count: 0
Column: sessions   | Type: INT64    | Compressed size:   567631 bytes
  Has statistics: True
  Min: 0
  Max: 500
  Null count: 0
Column: revenue    | Type: DOUBLE   | 

In [7]:
df.to_csv("data.csv", index=False)

In [8]:
import os
parquet_size = os.path.getsize('data.parquet')
parquet_size_KB = parquet_size/1024
csv_size = os.path.getsize('data.csv')
csv_size_KB = csv_size/1024
compression_ratio = csv_size / parquet_size
print(f"Parquet size: {parquet_size_KB:.2f} KB")
print(f"CSV size: {csv_size_KB:.2f} KB")
print(f"Compression ratio (CSV / Parquet): {compression_ratio:.2f}x")

Parquet size: 9504.63 KB
CSV size: 29124.35 KB
Compression ratio (CSV / Parquet): 3.06x


## Step 4: Why Parquet Metadata Matters (and CSV Does Not Have It)

When comparing Parquet and CSV, the most important difference is that **Parquet stores rich metadata**, while CSV stores almost none.

### 1. Schema Information

Parquet stores:
- Column names
- Data types (int, float, boolean, timestamp, etc.)
- Logical types (e.g., timestamp vs string)

CSV:
- Stores only raw text
- Does not store data types
- Requires type inference when reading

**Why it matters:**  
Query engines (like DuckDB) can immediately understand the structure of the data without scanning the entire file. CSV must be parsed and inferred first, which is slower and less reliable.

---

### 2. Row Groups

Parquet divides data into **row groups**.  
Each row group contains:
- A subset of rows
- Column chunks stored separately

CSV:
- Is a flat row-based file
- Has no internal structure for partial reading

**Why it matters:**  
Query engines can skip entire row groups if they are not relevant to a query. CSV must be scanned line by line from top to bottom.

---

### 3. Column-Level Statistics

For each column in each row group, Parquet stores statistics such as:
- Minimum value
- Maximum value
- Null count
- (Sometimes) distinct count

CSV:
- Stores no statistics
- Requires scanning all rows to compute them

**Why it matters:**  
This enables **predicate pushdown** and **row group pruning**.

Example:
If a query asks for:

```sql
SELECT * FROM data WHERE age > 70;

In [9]:
import pandas as pd
import time

start = time.time()

df_full = pd.read_parquet("data.parquet")

end = time.time()

print("Full read time:", round(end - start, 4), "seconds")

Full read time: 0.0181 seconds


In [10]:
start = time.time()

df_partial = pd.read_parquet(
    "data.parquet",
    columns=["age", "score"]
)

end = time.time()

print("Partial read time (2 columns):", round(end - start, 4), "seconds")

Partial read time (2 columns): 0.0022 seconds


In [11]:
start = time.time()

df_csv_partial = pd.read_csv(
    "data.csv",
    usecols=["age", "score"]
)

end = time.time()

print("CSV read (usecols) time:", round(end - start, 4), "seconds")

CSV read (usecols) time: 0.0792 seconds


## Step 4: Why Parquet Selective Reads Are Faster

Parquet is **columnar**, meaning each column is stored in separate chunks within row groups (as we saw in Task 1).  

- **Reading only 2 columns from Parquet** loads just those column chunks into memory → much less I/O → faster.  
- **Reading the full Parquet file** loads all columns → more I/O → slower.  
- **Reading columns from CSV** requires parsing the entire file first, even if we only need 2 columns → slowest of all.  

This explains the timing order we observed:  

In [12]:
import pyarrow.parquet as pq
import time

start = time.time()

table_filtered = pq.read_table(
    "data.parquet",
    filters=[("age", ">", 50)]
)

end = time.time()
time_pyarrow = round(end - start, 4)
print("Parquet read with filter (age > 50):", time_pyarrow, "seconds")
print("Number of rows read:", len(table_filtered))

Parquet read with filter (age > 50): 0.0131 seconds
Number of rows read: 237979


In [13]:

start = time.time()
df_full = pd.read_parquet("data.parquet")
df_filtered_post = df_full[df_full["age"] > 50]
end = time.time()
time_pandas = round(end - start, 4)
print("Full Parquet read + pandas filter time:", time_pandas, "seconds")
print("Number of rows after filter:", len(df_filtered_post))

Full Parquet read + pandas filter time: 0.0171 seconds
Number of rows after filter: 237979


In [14]:

print("Predicate pushdown (PyArrow) rows:", len(table_filtered))
print("Full read + pandas filter rows:", len(df_filtered_post))

print("Predicate pushdown (PyArrow) time:", round(time_pyarrow, 4), "sec")
print("Full read + pandas filter time:", round(time_pandas, 4), "sec")

Predicate pushdown (PyArrow) rows: 237979
Full read + pandas filter rows: 237979
Predicate pushdown (PyArrow) time: 0.0131 sec
Full read + pandas filter time: 0.0171 sec


## Step 3: Why Predicate Pushdown is Faster

Predicate pushdown means **applying a filter while reading the Parquet file**, instead of reading all rows first.

- Parquet stores **column-level statistics** (min, max, null count) for each row group.
- When a filter like `age > 50` is applied, Parquet can **skip entire row groups** that do not satisfy the condition.
- This reduces the **amount of data read from disk** and **memory usage**, making it much faster.

In contrast, pandas must **load the entire dataset** into memory first, then apply the filter.  


In [15]:
import duckdb
start= time.time()
result = duckdb.sql("SELECT * FROM 'data.parquet' WHERE age > 50").df()
end= time.time()

time_duckdb = round(end - start, 4)
print(f"DuckDB filtered query time: {time_duckdb} seconds")
print(f"Number of rows returned: {len(result)}")

DuckDB filtered query time: 0.0329 seconds
Number of rows returned: 237979


In [16]:
print("Execution time comparison (seconds):")
print(f"PyArrow predicate pushdown: {time_pyarrow}")
print(f"Pandas full read + filter: {time_pandas}")
print(f"DuckDB filtered query: {time_duckdb}")

print("\nNumber of rows returned:")
print(f"PyArrow: {len(df_filtered)}")
print(f"Pandas: {len(df_filtered_post)}")
print(f"DuckDB: {len(result)}")

Execution time comparison (seconds):
PyArrow predicate pushdown: 0.0131
Pandas full read + filter: 0.0171
DuckDB filtered query: 0.0329

Number of rows returned:


NameError: name 'df_filtered' is not defined

In [ ]:

df = pd.read_parquet("data.parquet")

start = time.time()

pandas_q1 = df.groupby("city").size().reset_index(name="count")

end = time.time()
time_pandas_q1 = round(end - start, 4)

print("Pandas Query 1 result:")
print(pandas_q1)
print("Execution time (pandas):", time_pandas_q1, "seconds")

In [ ]:
import duckdb
import time

start = time.time()

duckdb_q1 = duckdb.sql("""
    SELECT city, COUNT(*) as count
    FROM 'data.parquet'
    GROUP BY city
""").df()

end = time.time()
time_duckdb_q1 = round(end - start, 4)

print("\nDuckDB Query 1 result:")
print(duckdb_q1)
print("Execution time (DuckDB):", time_duckdb_q1, "seconds")

In [ ]:
start = time.time()
pandas_q2 = (
    df.groupby("city")["score"]
      .mean()
      .reset_index(name="avg_score")
      .sort_values("avg_score", ascending=False)
)
end = time.time()
time_pandas_q2 = round(end - start, 4)

print("Pandas Query 2 result:")
print(pandas_q2)
print("Execution time (pandas):", time_pandas_q2, "seconds")

In [ ]:

start = time.time()

duckdb_q2 = duckdb.sql("""
    SELECT city, AVG(score) as avg_score
    FROM 'data.parquet'
    GROUP BY city
    ORDER BY avg_score DESC
""").df()


end = time.time()
time_duckdb_q2 = round(end - start, 4)

print("\nDuckDB Query 2 result:")
print(duckdb_q2)
print("Execution time (DuckDB):", time_duckdb_q2, "seconds")

In [ ]:

start = time.time()
filtered = df[(df["active"]) & (df["score"] > 75)]
counts = filtered.groupby("city").size().reset_index(name="active_high_score")

totals = df.groupby("city").size().reset_index(name="total_users")

pandas_q3 = counts.merge(totals, on="city")
pandas_q3["pct_active_high_score"] = (
    pandas_q3["active_high_score"] / pandas_q3["total_users"] * 100
).round(2)

end = time.time()
time_pandas_q3 = round(end - start, 4)

print("Pandas Query 3 result:")
print(pandas_q3)
print("Execution time (pandas):", time_pandas_q3, "seconds")

In [ ]:
start = time.time()

duckdb_q3 = duckdb.sql("""
    SELECT 
        city,
        COUNT(*) AS active_high_score,
        COUNT(*) * 1.0 / (SELECT COUNT(*) FROM 'data.parquet' p2 WHERE p2.city = p1.city) * 100 AS pct_active_high_score
    FROM 'data.parquet' p1
    WHERE active = TRUE AND score > 75
    GROUP BY city
""").df()

end = time.time()
time_duckdb_q3 = round(end - start, 4)

print("\nDuckDB Query 3 result:")
print(duckdb_q3) 
print("Execution time (DuckDB):", time_duckdb_q3, "seconds")

In [ ]:
start = time.time()

df_sorted = df.sort_values(['city','score'],ascending=[True,False])
df_sorted['rank'] = df_sorted.groupby('city')['score'].rank(method='first',ascending=False)
pandas_q4 = df_sorted[df_sorted['rank']<=10].drop(columns=['rank'])
end = time.time()
time_pandas_q4 = round(end - start, 4)

print("Pandas Query 4 result (top 10 per city):")
print(pandas_q4.head(20))  
print("Execution time (pandas):", time_pandas_q4, "seconds")

In [ ]:

start = time.time()
duckdb_q4 = duckdb.sql("""
    select * from(
        select * ,
            rank() over (partition by city order by score desc) as rnk
        from 'data.parquet'
    ) sub 
    where rnk <=10
""").df()

end = time.time()
time_duckdb_q4 = round(end - start, 4)

print("\nDuckDB Query 4 result (top 10 per city):")
print(duckdb_q4.head(20))
print("Execution time (DuckDB):", time_duckdb_q4, "seconds")

In [ ]:

start = time.time()

pandas_q5 = (
    df.sort_values(["city", "user_id"])
        .assign(running_total = lambda x:x.groupby('city')['score'].cumsum())
)

end = time.time()
pandas_time_q5 = round(end - start, 4)

print("Pandas Q5 time:", pandas_time_q5)
print(pandas_q5.head())

In [ ]:
start = time.time()

duckdb_q5 = duckdb.sql("""
SELECT 
    user_id,
    city,
    score,
    sum(score) over(partition by city order by user_id rows between unbounded preceding and current row) AS running_total
FROM df
""").df()

end = time.time()
duckdb_time_q5 = round(end - start, 4)

print("DuckDB Q5 time:", duckdb_time_q5)
print(duckdb_q5.head())

### Comparison: Pandas vs DuckDB

**Ease of expressing queries**

For simple analytical tasks such as counting records per city or calculating the average score, **pandas** was very easy to use because its syntax is intuitive and designed for DataFrame manipulation. However, for more complex operations such as **window functions** (ranking users or computing running totals), **DuckDB SQL** was easier and more natural to express. SQL provides built-in constructs like `RANK()` and `OVER (PARTITION BY ...)`, which makes these queries more concise and readable compared to pandas.

**Performance comparison**

In general, **DuckDB was faster** than pandas for most queries, especially when performing aggregations and window functions. DuckDB uses a **vectorized execution engine and a query optimizer**, which allows it to process large datasets efficiently. Pandas also uses vectorized operations but does not have the same level of query optimization as a database engine.

**Where the difference mattered most**

The performance difference was most noticeable in the **more complex analytical queries**, particularly:
- **Top 10 users per city (ranking / window functions)**
- **Running totals partitioned by city**

These queries require sorting, grouping, and window calculations, which database engines like DuckDB handle very efficiently. For simpler operations such as counting rows or computing averages, the performance difference between pandas and DuckDB was smaller.

Overall, **pandas is very convenient for exploratory analysis**, while **DuckDB is better suited for complex analytical queries and large datasets**.

In [ ]:
import pandas as pd
import pyarrow as pa

df_arrow = pd.DataFrame({
    "user_id": [1, 2, 3, 4, 5],
    "city": ["Berlin", "Tokyo", "New York", "Paris", "London"],
    "score": [85.5, 92.0, 76.3, 88.1, 90.4],
    "active": [True, True, False, True, False]
})

print("Pandas DataFrame:")
print(df_arrow)

arrow_table = pa.Table.from_pandas(df_arrow)

print("\nArrow Table:")
print(arrow_table)



In [ ]:
print("\nArrow Schema:")
print(arrow_table.schema)

print("\nPandas dtypes:")
print(df_arrow.dtypes)

The Arrow schema is similar to pandas dtypes but uses a more explicit type system. 
For example, the `city` column appears as `str` in pandas but as `large_string` in Arrow, which is optimized for large string columns. 
Numeric types are also named differently (e.g., `float64` in pandas vs `double` in Arrow), but they represent the same data type.

Arrow also stores additional metadata about the pandas DataFrame (such as index information) so the data can be reconstructed correctly when converting back to pandas.

In [ ]:
pq.write_table(arrow_table, "arrow_data.parquet")
arrow_table_read = pq.read_table("arrow_data.parquet")


print("Arrow Table loaded from Parquet:")
print(arrow_table_read)

print("\nSchema:")
print(arrow_table_read.schema)

In [ ]:
arrow_table_to_pandas = arrow_table_read.to_pandas()
print(arrow_table_to_pandas.dtypes)
print()
print(df_arrow.dtypes)

In [ ]:
print("Data is identical:", arrow_table_to_pandas.equals(df_arrow))

In [ ]:
df_numpy = pd.read_parquet('arrow_data.parquet')
df_arrow_backend = pd.read_parquet('arrow_data.parquet',dtype_backend="pyarrow")

print("NumPy-backed dtypes:")
print(df_numpy.dtypes)

print("\nArrow-backed dtypes:")
print(df_arrow_backend.dtypes)

### The Role of Apache Arrow in the Modern Analytics Stack

Apache Arrow acts as a **bridge between storage and analytics**. Parquet files store data efficiently on disk in a **columnar format**, but to analyze that data, it needs to be loaded into memory. Arrow provides a **standardized in-memory columnar format** that multiple tools can use without copying or serialization.

This allows:  

- **pandas** to manipulate data interactively while keeping memory usage low.  
- **DuckDB** and other SQL engines to run vectorized, high-performance queries directly on the same in-memory data.  
- **Seamless interoperability** between tools: the same Arrow Table can be read, filtered, and aggregated by different engines efficiently.  

In short:



Arrow ensures that **data moves fast, uses memory efficiently, and avoids unnecessary conversions**, making modern local-first analytics much faster and more flexible.